In [9]:
import nd2
import numpy as np
import imageio.v3 as iio
from pathlib import Path
from datetime import date
from natsort import natsorted

In [7]:
# --- 1. Define Paths and Create Save Folder ---

# The folder containing your ND2 files
input_folder = Path(r"C:\Users\b1810\OneDrive - zonemail.clpccd.edu\abrain\LAB\10-20-25_Confocol") 

# The base directory where the converted images will be saved
base_save_path = Path(r"C:\Users\b1810\OneDrive - zonemail.clpccd.edu\abrain\LAB\10-20-25_Confocol\converted_images")

In [13]:
# Create a date-stamped subfolder (e.g., Converted_2025-11-13)
date_string = date.today().strftime("%Y-%m-%d")
save_folder = base_save_path / f"Converted_{date_string}"

# Create the folder if it doesn't exist
save_folder.mkdir(parents=True, exist_ok=True) 

print(f"Saving colorized channel JPEG files to: {save_folder}")

# --- 2. Find ND2 Files ---

# Find all ND2 files recursively within the input folder
nd2_files = natsorted(list(input_folder.glob("**/*.nd2")))

print(f"Found {len(nd2_files)} ND2 files to convert.")

def to_uint8(img):
    img = img.astype(np.float32)
    min_val = float(np.min(img))
    max_val = float(np.max(img))
    if max_val > min_val:
        return ((img - min_val) / (max_val - min_val) * 255).astype(np.uint8)
    return np.zeros_like(img, dtype=np.uint8)

# Color map used for channel tinting
channel_colors = [
    (255, 0, 0),   # red
    (0, 255, 0),   # green
    (0, 0, 255),   # blue
    (255, 255, 0), # yellow
    (255, 0, 255), # magenta
    (0, 255, 255), # cyan
]
color_names = ["red", "green", "blue", "yellow", "magenta", "cyan"]

# --- 3. Loop and Convert ---

for input_file_path in nd2_files:
    print(f"Converting: {input_file_path.name}...")
    
    try:
        # Read ND2 into a numpy array
        nd2_data = nd2.imread(str(input_file_path))
        data = np.asarray(nd2_data)

        # Reduce extra non-spatial dimensions while preserving channel information when possible
        while data.ndim > 3:
            data = data[0]

        if data.ndim == 2:
            # Single-channel image
            channels = data[np.newaxis, :, :]

        elif data.ndim == 3:
            # Case 1: channels-first (C, Y, X)
            if data.shape[0] <= 4 and data.shape[1] > 4 and data.shape[2] > 4:
                channels = data
            # Case 2: channels-last (Y, X, C)
            elif data.shape[2] <= 4 and data.shape[0] > 4 and data.shape[1] > 4:
                channels = np.moveaxis(data, -1, 0)
            else:
                # Likely a stack (e.g., Z, Y, X): create one projected channel
                channels = np.max(data, axis=0, keepdims=True)
                print("  -> No clear channel axis detected; saved max-projection as one channel")

        else:
            raise ValueError(f"Unsupported ND2 array shape: {data.shape}")

        # Save each channel as a separate colorized RGB JPEG
        n_channels = channels.shape[0]
        for ch_idx in range(n_channels):
            ch_img_uint8 = to_uint8(channels[ch_idx])

            # Pick tint color, cycling if channel count exceeds available colors
            color = channel_colors[ch_idx % len(channel_colors)]
            color_name = color_names[ch_idx % len(color_names)]

            rgb_img = np.zeros((ch_img_uint8.shape[0], ch_img_uint8.shape[1], 3), dtype=np.uint8)
            for c in range(3):
                if color[c] > 0:
                    rgb_img[:, :, c] = ch_img_uint8

            ch_output_path = save_folder / f"{input_file_path.stem}_ch{ch_idx + 1}_{color_name}.jpg"
            iio.imwrite(ch_output_path, rgb_img)

        print(f"  -> Saved {n_channels} colorized channel image(s)")
        
    except Exception as e:
        print(f"  -> ERROR converting {input_file_path.name}: {e}")

print("Conversion process complete.")

Saving colorized channel JPEG files to: C:\Users\b1810\OneDrive - zonemail.clpccd.edu\abrain\LAB\10-20-25_Confocol\converted_images\Converted_2026-03-24
Found 25 ND2 files to convert.
Converting: DMT female 4 CaST 1.nd2...
  -> Saved 3 colorized channel image(s)
Converting: DMT female 4 CaST 2.nd2...
  -> Saved 3 colorized channel image(s)
Converting: DMT female 4 CaST 3.nd2...
  -> Saved 3 colorized channel image(s)
Converting: DMT female 5 CaST 1.nd2...
  -> Saved 3 colorized channel image(s)
Converting: DMT female 5 CaST 2.nd2...
  -> Saved 3 colorized channel image(s)
Converting: DMT female 5 CaST 3.nd2...
  -> Saved 3 colorized channel image(s)
Converting: DMT female 5 CaST central 1.nd2...
  -> Saved 3 colorized channel image(s)
Converting: DMT female 5 CaST central 1_top layer.nd2...
  -> Saved 3 colorized channel image(s)
Converting: DMT female 5 CaST central 2.nd2...
  -> Saved 3 colorized channel image(s)
Converting: DMT female 5 CaST central 3.nd2...
  -> Saved 3 colorized c